In [1]:
import torch
import numpy as np
import cv2
import gradio as gr
from PIL import Image

from torchvision import transforms, models
import torch.nn as nn

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [3]:
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [4]:
# -------------------------------------------
# BUILD RESNET18 MODEL FOR TB SEVERITY
# -------------------------------------------

# Load pretrained ResNet18
model = models.resnet18(pretrained=True)

# Get number of input features of final layer
num_features = model.fc.in_features

# Replace classification layer with regression layer
# Output = 1 value (severity score)
model.fc = nn.Linear(num_features, 1)

# Move model to GPU if available
model = model.to(device)

# Print model summary
print(model)

C:\Users\Midhunraj N P\anaconda3\envs\ai\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Midhunraj N P\anaconda3\envs\ai\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
# -------------------------------------------
# LOAD TRAINED MODEL
# -------------------------------------------

model.load_state_dict(
    torch.load(
        "models/severity_resnet18_tbx11k.pth",
        map_location=device
    )
)

model.eval()

print("Model loaded successfully!")

Model loaded successfully!


C:\Users\Midhunraj N P\AppData\Local\Temp\ipykernel_20008\4257763387.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


In [6]:
# -------------------------------------------
# TARGET LAYER FOR GRAD-CAM
# -------------------------------------------

target_layer = model.layer4[-1]

# -------------------------------------------
# HOOKS FOR GRAD-CAM
# -------------------------------------------

gradients = None
activations = None

def backward_hook(module, grad_input, grad_output):
    global gradients
    gradients = grad_output[0]

def forward_hook(module, input, output):
    global activations
    activations = output

# Register hooks
target_layer.register_forward_hook(forward_hook)
target_layer.register_full_backward_hook(backward_hook)

In [7]:
# -------------------------------------------
# GRAD-CAM GENERATION FUNCTION
# -------------------------------------------

def generate_gradcam(image_tensor):

    global gradients, activations

    image_tensor = image_tensor.unsqueeze(0).to(device)

    output = model(image_tensor)

    model.zero_grad()
    output.backward()

    grad = gradients.cpu().detach().numpy()[0]
    act = activations.cpu().detach().numpy()[0]

    weights = np.mean(grad, axis=(1, 2))

    cam = np.zeros(act.shape[1:], dtype=np.float32)

    for i, w in enumerate(weights):
        cam += w * act[i]

    cam = np.maximum(cam, 0)

    cam = cv2.resize(cam, (224, 224))

    cam = cam / cam.max()

    return cam

In [8]:
# -------------------------------------------
# PREDICT SEVERITY
# -------------------------------------------

def predict_severity(image):

    image = image.convert("RGB")

    tensor = val_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model(tensor)

    return prediction.item()

In [9]:
# -------------------------------------------
# SINGLE IMAGE TB ANALYSIS
# -------------------------------------------

def predict_tb(image):

    # Convert image
    image_pil = Image.fromarray(image).convert("RGB")

    # Severity prediction
    severity = predict_severity(image_pil)

    # GradCAM
    tensor = val_transform(image_pil)

    cam = generate_gradcam(tensor)

    original = np.array(image_pil.resize((224,224)))

    heatmap = np.uint8(255 * cam)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    overlay = cv2.addWeighted(
        original,
        0.6,
        heatmap,
        0.4,
        0
    )

    result = f"Predicted Severity Score: {severity:.3f}"

    return result, overlay

In [10]:
# -------------------------------------------
# LONGITUDINAL ANALYSIS
# -------------------------------------------

def longitudinal_tb(images):

    if images is None or len(images) == 0:
        return "Please upload images."

    severities = []

    for img in images:

        if isinstance(img, tuple):
            img = img[0]

        if isinstance(img, str):
            image = Image.open(img).convert("RGB")
        else:
            image = Image.fromarray(img).convert("RGB")

        severity = predict_severity(image)

        severities.append(severity)

    result = ""

    for i, s in enumerate(severities):
        result += f"Visit {i+1}: Severity {s:.3f}\n"

    return result

In [11]:
# -------------------------------------------
# SINGLE PREDICTION TAB
# -------------------------------------------

single_interface = gr.Interface(
    fn=predict_tb,
    inputs=gr.Image(),
    outputs=[
        gr.Textbox(label="Prediction"),
        gr.Image(label="Grad-CAM")
    ],
    title="TB Severity Prediction"
)

# -------------------------------------------
# LONGITUDINAL TAB
# -------------------------------------------

longitudinal_interface = gr.Interface(
    fn=longitudinal_tb,
    inputs=gr.Gallery(label="Upload Multiple X-rays"),
    outputs=gr.Textbox(label="Severity Scores"),
    title="TB Longitudinal Analysis"
)

# -------------------------------------------
# COMBINED APP
# -------------------------------------------

app = gr.TabbedInterface(
    [single_interface, longitudinal_interface],
    ["Single Prediction", "Longitudinal Analysis"]
)

In [12]:
app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


C:\Users\Midhunraj N P\anaconda3\envs\ai\lib\site-packages\torch\autograd\graph.py:825: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
